# §12.7.4 — 길이에 따른 학습·추론 시간과 메모리의 측정

> 딥러닝 교재 · 3부 12장 7절 4항 (🐍)
> 선행: §12.7.1(순차성의 정의) · §12.7.3(학습과 추론의 비대칭) · §12.7.5(추론에서의 이점)

## 이 노트북이 답하는 질문

1. **학습 한 걸음의 시간은 길이 $T$와 함께 어떻게 자라는가?** 순환(순차)과 어텐션(병렬)의 기울기를 잰다.
2. **생성 토큰당 시간은?** 순환의 상수와 어텐션(KV 캐시)의 선형 성장을 확인한다.
3. **생성 중 보관해야 하는 메모리는?** 고정 상태 대 길이 비례 캐시.

**예상 실행 시간** CPU 약 2분 (`FAST = True`이면 약 1분).

⚠︎ 이 노트북이 재는 것은 단일 스레드 NumPy/CPU의 벽시계 시간이다. 어텐션의 "병렬성"은
여기서 시간축 전체를 한 번의 큰 행렬 곱(BLAS)으로 처리하는 벡터화로 나타나고, 순환의
"순차성"은 파이썬 루프로 나타난다. GPU에서는 같은 비대칭이 수천 코어의 병렬화로 증폭될
뿐, $O(T)$ 순차 대 $O(T^2)$ 병렬이라는 점근 구조 자체는 동일하다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 같은 상태 크기의 두 모델

상태 크기(폭) $m$을 맞춘 두 시퀀스 모델을 준비한다. 학습 시간 측정에는 손실의 값이
중요하지 않으므로, 모든 위치에 판독을 두고 더미 제곱 손실로 완전한 순전파+역전파의
벽시계 시간만 잰다.

* **순환** — $h_t=\tanh(W_h h_{t-1}+W_x x_t+b)$. 순전파도 역전파도 $T$번의 순차 루프다.
* **어텐션** — 단일 헤드 인과 자기 어텐션 층 하나:
  $Q=XW_Q,\ K=XW_K,\ V=XW_V$, $\alpha=\operatorname{softmax}(QK^\top/\sqrt m + \text{인과 마스크})$,
  $Y=(\alpha V)W_O$. 시간축 전체가 몇 번의 큰 행렬 곱으로 한 번에 처리된다.
  (기호는 §13.1의 것을 미리 빌린다.)

배치는 $B=2$로 작게 둔다. 배치가 크면 순환도 배치축의 벡터화로 스텝당 오버헤드를
나눠 갚으므로, **시간축**의 순차/병렬 대비가 흐려지기 때문이다. 순차 루프의 비용이
그대로 드러나는 작은 배치가 이 실험의 관심사에 맞다.

In [ ]:
def rnn_train_step(T, m=64, B=2, rn=None):
    # 순전파 + 전 시각 판독 손실 + 완전한 BPTT. 시간만 잰다.
    rn = rn or np.random.default_rng(0)
    Wh = rn.standard_normal((m, m)) * (0.9 / np.sqrt(m))
    Wx = rn.standard_normal((m, m)) / np.sqrt(m)
    b = np.zeros(m); U = rn.standard_normal((m, m)) / np.sqrt(m)
    X = rn.standard_normal((B, T, m))
    t0 = time.perf_counter()
    H = np.zeros((B, T + 1, m)); Z = np.zeros((B, T, m))
    for t in range(T):                                   # 순차 루프 (전진)
        Z[:, t] = H[:, t] @ Wh + X[:, t] @ Wx + b
        H[:, t + 1] = np.tanh(Z[:, t])
    Yh = H[:, 1:] @ U
    dY = Yh / (B * T * m)                                # 더미 제곱 손실의 기울기
    dH_out = dY @ U.T
    gWh = np.zeros_like(Wh); gWx = np.zeros_like(Wx); gb = np.zeros_like(b)
    delta = dH_out[:, T - 1]
    for t in range(T - 1, -1, -1):                       # 순차 루프 (후진)
        dz = delta * (1 - np.tanh(Z[:, t]) ** 2)
        gWh += H[:, t].T @ dz; gWx += X[:, t].T @ dz; gb += dz.sum(0)
        delta = dz @ Wh.T + (dH_out[:, t - 1] if t > 0 else 0)
    gU = 0  # 판독 기울기는 위에서 함께 처리한 셈 치고 생략 (시간 지배항 아님)
    return time.perf_counter() - t0

def attn_train_step(T, m=64, B=2, rn=None):
    rn = rn or np.random.default_rng(0)
    Wq, Wk, Wv, Wo = (rn.standard_normal((m, m)) / np.sqrt(m) for _ in range(4))
    X = rn.standard_normal((B, T, m))
    mask = np.triu(np.full((T, T), -np.inf), k=1)        # 인과 마스크
    t0 = time.perf_counter()
    Q = X @ Wq; K = X @ Wk; V = X @ Wv                   # 시간축 전체를 한 번에
    S_ = Q @ K.transpose(0, 2, 1) / np.sqrt(m) + mask
    S_ -= S_.max(axis=2, keepdims=True)
    al = np.exp(S_); al /= al.sum(axis=2, keepdims=True)
    C = al @ V
    Y = C @ Wo
    dY = Y / (B * T * m)
    dC = dY @ Wo.T; gWo = np.einsum('btm,btn->mn', C, dY)
    dal = dC @ V.transpose(0, 2, 1)
    dV = al.transpose(0, 2, 1) @ dC
    dS = al * (dal - (al * dal).sum(axis=2, keepdims=True))
    dQ = dS @ K / np.sqrt(m)
    dK = dS.transpose(0, 2, 1) @ Q / np.sqrt(m)
    gWq = np.einsum('btm,btn->mn', X, dQ)
    gWk = np.einsum('btm,btn->mn', X, dK)
    gWv = np.einsum('btm,btn->mn', X, dV)
    return time.perf_counter() - t0

def timeit_min(fn, reps=3, **kw):
    return min(fn(**kw) for _ in range(reps))

T_GRID = [8, 32, 128, 512, 2048] if FAST else [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
t_rnn, t_att = [], []
for T in T_GRID:
    t_rnn.append(timeit_min(rnn_train_step, T=T))
    t_att.append(timeit_min(attn_train_step, T=T))
    print(f"T={T:5d}:  순환 {t_rnn[-1]*1e3:8.1f} ms   어텐션 {t_att[-1]*1e3:8.1f} ms")

sl_rnn = np.polyfit(np.log(T_GRID), np.log(t_rnn), 1)[0]
sl_att_lo = np.polyfit(np.log(T_GRID[:5]), np.log(t_att[:5]), 1)[0]
sl_att_hi = np.polyfit(np.log(T_GRID[-3:]), np.log(t_att[-3:]), 1)[0]
print(f"로그–로그 기울기:  순환 {sl_rnn:.2f} | 어텐션(짧은 구간) {sl_att_lo:.2f} → (긴 구간) {sl_att_hi:.2f}")

---
## 2. 생성 — 토큰당 시간과 보관 메모리

한 토큰씩 생성하는 상황을 흉내 낸다. 순환은 상태 $h$ 하나를 갱신하면 끝이고($O(1)$/토큰),
어텐션은 지금까지의 $K,V$ 캐시 전부와 내적해야 하므로 문맥 길이 $t$에 비례한다.
메모리도 마찬가지: 순환은 $m$개 실수가 전부, 어텐션 캐시는 $2\,t\,m$개로 자란다.

In [ ]:
def gen_measure(T_max, m=128, bucket=64, reps=3 if not FAST else 2):
    rn = np.random.default_rng(0)
    Wh = rn.standard_normal((m, m)) * (0.9 / np.sqrt(m))
    Wx = rn.standard_normal((m, m)) / np.sqrt(m)
    Wq, Wk, Wv = (rn.standard_normal((m, m)) / np.sqrt(m) for _ in range(3))
    n_b = T_max // bucket
    tr = np.zeros((reps, n_b)); ta = np.zeros((reps, n_b))
    mem_r = np.zeros(n_b); mem_a = np.zeros(n_b)
    for r in range(reps):
        # 순환 생성
        h = np.zeros(m); x = rn.standard_normal(m)
        for bidx in range(n_b):
            t0 = time.perf_counter()
            for _ in range(bucket):
                h = np.tanh(h @ Wh + x @ Wx)
            tr[r, bidx] = (time.perf_counter() - t0) / bucket
            mem_r[bidx] = h.nbytes
        # 어텐션 생성 (KV 캐시)
        Kc = np.zeros((0, m)); Vc = np.zeros((0, m)); x = rn.standard_normal(m)
        for bidx in range(n_b):
            t0 = time.perf_counter()
            for _ in range(bucket):
                k_new = (x @ Wk)[None, :]; v_new = (x @ Wv)[None, :]
                Kc = np.concatenate([Kc, k_new]); Vc = np.concatenate([Vc, v_new])
                q = x @ Wq
                e = Kc @ q / np.sqrt(m); e -= e.max()
                al = np.exp(e); al /= al.sum()
                c = al @ Vc
            ta[r, bidx] = (time.perf_counter() - t0) / bucket
            mem_a[bidx] = Kc.nbytes + Vc.nbytes
    ctx = (np.arange(n_b) + 1) * bucket
    return ctx, tr.min(0), ta.min(0), mem_r, mem_a

T_MAX = 1024 if FAST else 2048
ctx, tok_r, tok_a, mem_r, mem_a = gen_measure(T_MAX)
print(f"문맥 {ctx[0]}:   순환 {tok_r[0]*1e6:6.1f} µs/토큰 | 어텐션 {tok_a[0]*1e6:6.1f} µs/토큰")
print(f"문맥 {ctx[-1]}: 순환 {tok_r[-1]*1e6:6.1f} µs/토큰 | 어텐션 {tok_a[-1]*1e6:6.1f} µs/토큰")
print(f"메모리(문맥 {ctx[-1]}): 순환 {mem_r[-1]/1024:.1f} KiB | 어텐션 캐시 {mem_a[-1]/2**20:.2f} MiB")

---
## 3. 교재 그림 — fig_12_7_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 학습 스텝 시간
ax = axes[0]
ax.loglog(T_GRID, np.array(t_rnn) * 1e3, 'o-', color=CB[4], ms=5,
          label=lab(f'순환 (기울기 {sl_rnn:.2f})', f'RNN (slope {sl_rnn:.2f})'))
ax.loglog(T_GRID, np.array(t_att) * 1e3, 's-', color=CB[5], ms=5,
          label=lab(f'어텐션 ({sl_att_lo:.1f} → {sl_att_hi:.1f})',
                    f'attention ({sl_att_lo:.1f}→{sl_att_hi:.1f})'))
ref = np.array(T_GRID, float)
ax.loglog(ref, ref / ref[0] * t_rnn[0] * 1e3, ':', color='0.6', lw=0.8)
ax.text(ref[-2], ref[-2] / ref[0] * t_rnn[0] * 1e3 * 0.5, '$\\propto T$', fontsize=8, color='0.4')
ax.loglog(ref[3:], (ref[3:] / ref[3]) ** 2 * t_att[3] * 1e3, '--', color='0.6', lw=0.8)
ax.text(ref[-2], (ref[-2] / ref[3]) ** 2 * t_att[3] * 1e3 * 1.6, '$\\propto T^2$', fontsize=8, color='0.4')
ax.set_xlabel(lab('시퀀스 길이 $T$', 'sequence length $T$'))
ax.set_ylabel(lab('학습 스텝 시간 (ms)', 'train step time (ms)'))
ax.set_title(lab('(a) 학습: 순차 $O(T)$ 대 병렬 $O(T^2)$', '(a) training step'), fontsize=10)
ax.legend(fontsize=8, loc='upper left')

# (b) 생성 토큰당 시간
ax = axes[1]
ax.semilogy(ctx, tok_r * 1e6, '-', color=CB[4], lw=1.4, label=lab('순환', 'RNN'))
ax.semilogy(ctx, tok_a * 1e6, '-', color=CB[5], lw=1.4, label=lab('어텐션 (KV 캐시)', 'attention'))
ax.set_xlabel(lab('현재 문맥 길이 $t$', 'context length $t$'))
ax.set_ylabel(lab('토큰당 생성 시간 (µs)', 'time per token (µs)'))
ax.set_title(lab('(b) 생성: 상수 대 선형', '(b) generation'), fontsize=10)
ax.legend(fontsize=8)

# (c) 생성 시 보관 메모리
ax = axes[2]
ax.semilogy(ctx, mem_r / 1024, '-', color=CB[4], lw=1.4, label=lab('순환: 상태 $h$', 'RNN state'))
ax.semilogy(ctx, mem_a / 1024, '-', color=CB[5], lw=1.4, label=lab('어텐션: $K,V$ 캐시', 'KV cache'))
ax.set_xlabel(lab('현재 문맥 길이 $t$', 'context length $t$'))
ax.set_ylabel(lab('보관 메모리 (KiB)', 'memory (KiB)'))
ax.set_title(lab('(c) 메모리: 수평선 대 $2tm$', '(c) memory'), fontsize=10)
ax.legend(fontsize=8)

# (d) 네 조합 요약
ax = axes[3]; ax.axis('off')
rows = [
    [lab('', ''), lab('학습 (전체 열)', 'train'), lab('생성 (토큰당)', 'generate')],
    [lab('순환', 'RNN'),
     lab(f'$O(T)$, 병렬화 불가\n(T={T_GRID[-1]}: {t_rnn[-1]*1e3:.0f} ms)', 'seq'),
     lab(f'$O(1)$ 시간·메모리\n({tok_r[-1]*1e6:.0f} µs, {mem_r[-1]/1024:.0f} KiB)', 'const')],
    [lab('어텐션', 'attn'),
     lab(f'$O(T^2)$, 병렬화 가능\n(단일 스레드: {t_att[-1]*1e3:.0f} ms)', 'par'),
     lab(f'$O(t)$ 시간·메모리\n({tok_a[-1]*1e6:.0f} µs, {mem_a[-1]/2**20:.1f} MiB)', 'lin')],
]
colors = [['none', '#F3F3F3', '#F3F3F3'],
          ['#F3F3F3', '#FDEBD9', '#E8F7F0'],
          ['#F3F3F3', '#E8F7F0', '#FDEBD9']]
for i in range(3):
    for j in range(3):
        ax.add_patch(plt.Rectangle((j * 3.3, 6 - i * 2.4 - 2.2), 3.1, 2.1,
                                   fc=colors[i][j], ec='0.6', lw=0.7,
                                   transform=ax.transData))
        ax.text(j * 3.3 + 1.55, 6 - i * 2.4 - 1.15, rows[i][j], ha='center', va='center',
                fontsize=8.5)
ax.set_xlim(-0.2, 10.1); ax.set_ylim(-1.3, 6.4)
ax.text(4.95, -0.9, lab('초록 = 구조적 승자. 학습의 승부는 병렬 하드웨어에서 갈리고(§12.7.3), 생성은 상수 상태가 이긴다',
                        'green = structural winner per phase'), ha='center', fontsize=8)
ax.set_title(lab('(d) (학습, 생성) × (순환, 어텐션) 요약', '(d) summary'), fontsize=10)

save_book_fig(fig, 'fig_12_7_4')
plt.show()

> ### 읽는 법
>
> (a) 순환의 학습 시간은 로그–로그에서 기울기 1의 직선 — 순차 루프의 $O(T)$다.
> 어텐션은 두 얼굴을 보인다. 짧은 구간에서는 시간축 벡터화 덕에 순환보다 **빠르지만**
> (순환은 스텝마다 순차 오버헤드를 내는데 어텐션은 큰 행렬 곱 몇 번이 전부다),
> $T^2$ 항이 지배하는 구간부터 기울기 2로 꺾이며 추월당한다. **병렬화 가능은 공짜가
> 아니라 $O(T^2)$과의 교환이다**(§12.7.3).
> (b) 생성에서는 처지가 뒤집힌다. 순환의 토큰당 시간은 수평선, 어텐션은 문맥 길이에
> 선형으로 자란다. (c) 메모리도 같은 그림 — 상태 하나 대 $2tm$ 캐시.
> (d) 어느 쪽도 전면 우위가 아니다. 학습은 병렬(어텐션)이, 대화형 생성은 상수 상태
> (순환)가 이긴다. 이 표의 오른쪽 아래 칸을 되찾으려는 시도가 §13.7(효율 어텐션)과
> 14장(상태공간 모델)의 동력이 된다(§12.7.5).

---
## 4. 자기 점검

1. (a)에서 어텐션 곡선이 기울기 2로 꺾이는 $T$는 대략 얼마인가? $O(Tm^2)$ 사영 항과 $O(T^2m)$ 점수 항이 같아지는 $T\approx m$ 근방과 비교하라.
2. (b)의 순환 곡선이 완전한 수평이 아니라면 무엇 때문일 수 있는가? (힌트: 캐시 계층.)
3. 배치 $B$를 키우면 (a)의 두 곡선은 각각 어떻게 움직이는가? 실행해 보라.
4. 어텐션 생성의 토큰당 시간을 문맥과 무관하게 만들려면 무엇을 포기해야 하는가? §13.9(슬라이딩 윈도), §14장(상태공간)과 연결하라.

## 5. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `m` | 1·2절 | 64/128 | 교차 지점 $T\sim m$의 이동 |
| `T_GRID` | 1절 | ≤1024 | 더 긴 열의 점근 |
| `bucket` | 2절 | 64 | 측정 잡음 |
| `B` | 1절 | 16 | 배치 병렬성 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")